<a href="https://colab.research.google.com/github/MLfinal/Walmart-Recruiting---Store-Sales-Forecasting/blob/dev/models/deep_learning/DLinear/model_experiment_DLinear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DLinear experiment v4 — Store/Dept identity embeddings

Previous 39-week validation results:
- seasonal naive WMAE: `1604.27`
- baseline DLinear, 52w: `1523.21`
- v1, 52w + Store-Dept series calibration: `1506.28` best so far
- v2, free calendar branch: `1961.45` failed
- v3, gated calendar branch: `1511.97`

v4 removes calendar features because v3 did not beat v1. The next controlled improvement is better identity modeling:
- keep `input_weeks=52`;
- keep DLinear trend/seasonal core;
- keep Store-Dept `series_bias`;
- add separate Store and Dept embeddings;
- map embeddings to a 39-week horizon adjustment.

The test question is: does decomposing identity into Store and Dept embeddings improve over a single Store-Dept calibration vector?

In [ ]:
%pip install -q "torch>=2.3,<3" "wandb>=0.19,<1" "pandas>=2.2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4"

In [ ]:
from __future__ import annotations

import json
import math
import platform
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import wandb

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pd.set_option("display.max_columns", 100)
print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
})

In [ ]:
CONFIG = {
    "seed": SEED,
    "validation_weeks": 39,
    "input_weeks": 52,
    "holiday_weight": 5.0,
    "batch_size": 512,
    "epochs": 90,
    "learning_rate": 6e-4,
    "weight_decay": 2e-4,
    "series_bias_weight_decay": 1e-3,
    "identity_weight_decay": 1e-4,
    "patience": 14,
    "num_workers": 2,
    "clip_grad_norm": 1.0,
    "moving_avg_kernel": 25,
    "store_embedding_dim": 8,
    "dept_embedding_dim": 12,
    "identity_hidden_dim": 32,
    "use_series_calibration": True,
    "baseline_dlinear_39w_wmae": 1523.209716796875,
    "dlinear_v1_calibration_wmae": 1506.282470703125,
    "dlinear_v2_calendar_wmae": 1961.4508056640625,
    "dlinear_v3_gated_calendar_wmae": 1511.9732666015625,
    "seasonal_naive_wmae_reference": 1604.2697073319134,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "wandb_project": "Walmart-Recruiting---Store-Sales-Forecasting",
    "wandb_entity": "kende23-n-a",
    "wandb_group": "dlinear-experiments",
}

DATA_DIR = Path("/content/drive/MyDrive/walmart_competition_data")
OUTPUT_DIR = Path("/content/artifacts/dlinear_experiment_v4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"Not running in Colab or Drive unavailable: {exc}")

## Load data and build weekly panel

In [ ]:
train_raw = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["Date"])
test_raw = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["Date"])

required_train = {"Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"}
required_test = {"Store", "Dept", "Date", "IsHoliday"}
missing_train = required_train.difference(train_raw.columns)
missing_test = required_test.difference(test_raw.columns)
if missing_train or missing_test:
    raise ValueError({"missing_train": sorted(missing_train), "missing_test": sorted(missing_test)})

train_raw = train_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
test_raw = test_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

all_train_dates = pd.Index(sorted(train_raw["Date"].unique()), name="Date")
test_dates = pd.Index(sorted(test_raw["Date"].unique()), name="Date")
val_dates = all_train_dates[-CONFIG["validation_weeks"]:]
fit_dates = all_train_dates[:-CONFIG["validation_weeks"]]
split_pos = len(fit_dates)

if CONFIG["input_weeks"] + CONFIG["validation_weeks"] > split_pos:
    raise ValueError(
        f"input_weeks + validation_weeks must be <= pre-validation history. "
        f"Got {CONFIG['input_weeks']} + {CONFIG['validation_weeks']} > {split_pos}."
    )

sales_panel = (
    train_raw.pivot_table(index=["Store", "Dept"], columns="Date", values="Weekly_Sales", aggfunc="sum")
    .reindex(columns=all_train_dates)
    .fillna(0.0)
    .sort_index()
)

holiday_by_date = (
    train_raw[["Date", "IsHoliday"]]
    .drop_duplicates("Date")
    .set_index("Date")
    .reindex(all_train_dates)["IsHoliday"]
    .fillna(False)
    .astype(bool)
)

series_frame = pd.DataFrame(sales_panel.index.tolist(), columns=["Store", "Dept"])
store_values = sorted(series_frame["Store"].unique())
dept_values = sorted(series_frame["Dept"].unique())
store_to_idx = {int(v): i for i, v in enumerate(store_values)}
dept_to_idx = {int(v): i for i, v in enumerate(dept_values)}
series_store_idx = series_frame["Store"].map(store_to_idx).to_numpy(dtype=np.int64)
series_dept_idx = series_frame["Dept"].map(dept_to_idx).to_numpy(dtype=np.int64)

values = sales_panel.to_numpy(dtype=np.float32)
holiday_flags = holiday_by_date.to_numpy(dtype=bool)

print({
    "n_series": len(sales_panel),
    "n_stores": len(store_values),
    "n_depts": len(dept_values),
    "n_dates": len(all_train_dates),
    "fit_range": (str(fit_dates.min().date()), str(fit_dates.max().date())),
    "validation_range": (str(val_dates.min().date()), str(val_dates.max().date())),
    "train_windows_per_series": split_pos - CONFIG["input_weeks"] - CONFIG["validation_weeks"] + 1,
    "test_horizon": len(test_dates),
})

## Metric and datasets

`series_idx` is used for the Store-Dept calibration vector. `store_idx` and `dept_idx` are used for separate identity embeddings.

In [ ]:
def wmae(y_true: np.ndarray, y_pred: np.ndarray, is_holiday: np.ndarray, holiday_weight: float = 5.0) -> float:
    weights = np.where(np.asarray(is_holiday, dtype=bool), holiday_weight, 1.0)
    return float(np.sum(weights * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(weights))


class WindowDataset(Dataset):
    def __init__(self, values: np.ndarray, holiday_flags: np.ndarray, series_store_idx: np.ndarray, series_dept_idx: np.ndarray, input_len: int, pred_len: int, end_pos: int):
        self.values = values.astype(np.float32)
        self.holiday_flags = holiday_flags.astype(bool)
        self.series_store_idx = series_store_idx.astype(np.int64)
        self.series_dept_idx = series_dept_idx.astype(np.int64)
        self.input_len = int(input_len)
        self.pred_len = int(pred_len)
        self.end_pos = int(end_pos)
        self.index = []
        max_start = self.end_pos - self.input_len - self.pred_len
        if max_start < 0:
            raise ValueError("Not enough history for configured windows.")
        for series_idx in range(self.values.shape[0]):
            for start in range(max_start + 1):
                self.index.append((series_idx, start))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx: int):
        series_idx, start = self.index[idx]
        x = self.values[series_idx, start:start + self.input_len]
        y = self.values[series_idx, start + self.input_len:start + self.input_len + self.pred_len]
        mean = x.mean(dtype=np.float64).astype(np.float32)
        std = x.std(dtype=np.float64).astype(np.float32)
        std = np.float32(max(float(std), 1.0))
        target_positions = np.arange(start + self.input_len, start + self.input_len + self.pred_len)
        weights = np.where(self.holiday_flags[target_positions], CONFIG["holiday_weight"], 1.0).astype(np.float32)
        return {
            "x": torch.from_numpy(((x - mean) / std)[:, None]),
            "y": torch.from_numpy((y - mean) / std),
            "weights": torch.from_numpy(weights),
            "mean": torch.tensor(mean, dtype=torch.float32),
            "std": torch.tensor(std, dtype=torch.float32),
            "series_idx": torch.tensor(series_idx, dtype=torch.long),
            "store_idx": torch.tensor(self.series_store_idx[series_idx], dtype=torch.long),
            "dept_idx": torch.tensor(self.series_dept_idx[series_idx], dtype=torch.long),
        }


class ValidationDataset(Dataset):
    def __init__(self, values: np.ndarray, holiday_flags: np.ndarray, series_store_idx: np.ndarray, series_dept_idx: np.ndarray, input_len: int, pred_len: int, split_pos: int):
        self.values = values.astype(np.float32)
        self.holiday_flags = holiday_flags.astype(bool)
        self.series_store_idx = series_store_idx.astype(np.int64)
        self.series_dept_idx = series_dept_idx.astype(np.int64)
        self.input_len = int(input_len)
        self.pred_len = int(pred_len)
        self.split_pos = int(split_pos)

    def __len__(self):
        return self.values.shape[0]

    def __getitem__(self, series_idx: int):
        start = self.split_pos - self.input_len
        x = self.values[series_idx, start:self.split_pos]
        y = self.values[series_idx, self.split_pos:self.split_pos + self.pred_len]
        mean = x.mean(dtype=np.float64).astype(np.float32)
        std = x.std(dtype=np.float64).astype(np.float32)
        std = np.float32(max(float(std), 1.0))
        target_positions = np.arange(self.split_pos, self.split_pos + self.pred_len)
        weights = np.where(self.holiday_flags[target_positions], CONFIG["holiday_weight"], 1.0).astype(np.float32)
        return {
            "x": torch.from_numpy(((x - mean) / std)[:, None]),
            "y": torch.from_numpy((y - mean) / std),
            "weights": torch.from_numpy(weights),
            "mean": torch.tensor(mean, dtype=torch.float32),
            "std": torch.tensor(std, dtype=torch.float32),
            "series_idx": torch.tensor(series_idx, dtype=torch.long),
            "store_idx": torch.tensor(self.series_store_idx[series_idx], dtype=torch.long),
            "dept_idx": torch.tensor(self.series_dept_idx[series_idx], dtype=torch.long),
        }

## Model: DLinear + series calibration + Store/Dept embeddings

In [ ]:
class MovingAverage(nn.Module):
    def __init__(self, kernel_size: int):
        super().__init__()
        self.kernel_size = int(kernel_size)
        self.avg = nn.AvgPool1d(kernel_size=self.kernel_size, stride=1, padding=0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pad_left = (self.kernel_size - 1) // 2
        pad_right = self.kernel_size - 1 - pad_left
        front = x[:, :1, :].repeat(1, pad_left, 1)
        end = x[:, -1:, :].repeat(1, pad_right, 1)
        x_pad = torch.cat([front, x, end], dim=1)
        return self.avg(x_pad.permute(0, 2, 1)).permute(0, 2, 1)


class SeriesDecomposition(nn.Module):
    def __init__(self, kernel_size: int):
        super().__init__()
        self.moving_average = MovingAverage(kernel_size)

    def forward(self, x: torch.Tensor):
        trend = self.moving_average(x)
        seasonal = x - trend
        return seasonal, trend


class DLinearIdentity(nn.Module):
    def __init__(self, seq_len: int, pred_len: int, n_series: int, n_stores: int, n_depts: int, store_embedding_dim: int, dept_embedding_dim: int, identity_hidden_dim: int, moving_avg_kernel: int = 25, use_series_calibration: bool = True):
        super().__init__()
        self.seq_len = int(seq_len)
        self.pred_len = int(pred_len)
        self.decomposition = SeriesDecomposition(moving_avg_kernel)
        self.linear_seasonal = nn.Linear(self.seq_len, self.pred_len)
        self.linear_trend = nn.Linear(self.seq_len, self.pred_len)
        self.series_bias = nn.Embedding(n_series, self.pred_len) if use_series_calibration else None
        self.store_embedding = nn.Embedding(n_stores, store_embedding_dim)
        self.dept_embedding = nn.Embedding(n_depts, dept_embedding_dim)
        self.identity_head = nn.Sequential(
            nn.Linear(store_embedding_dim + dept_embedding_dim, identity_hidden_dim),
            nn.ReLU(),
            nn.Linear(identity_hidden_dim, self.pred_len),
        )
        self._init_weights()

    def _init_weights(self):
        nn.init.constant_(self.linear_seasonal.weight, 1.0 / self.seq_len)
        nn.init.constant_(self.linear_trend.weight, 1.0 / self.seq_len)
        nn.init.zeros_(self.linear_seasonal.bias)
        nn.init.zeros_(self.linear_trend.bias)
        if self.series_bias is not None:
            nn.init.zeros_(self.series_bias.weight)
        nn.init.normal_(self.store_embedding.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.dept_embedding.weight, mean=0.0, std=0.02)
        for module in self.identity_head.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor, series_idx: torch.Tensor, store_idx: torch.Tensor, dept_idx: torch.Tensor) -> torch.Tensor:
        seasonal, trend = self.decomposition(x)
        seasonal = seasonal.permute(0, 2, 1)
        trend = trend.permute(0, 2, 1)
        out = self.linear_seasonal(seasonal) + self.linear_trend(trend)
        out = out.permute(0, 2, 1).squeeze(-1)
        if self.series_bias is not None:
            out = out + self.series_bias(series_idx)
        identity_vector = torch.cat([self.store_embedding(store_idx), self.dept_embedding(dept_idx)], dim=1)
        identity_adjustment = self.identity_head(identity_vector)
        return out + identity_adjustment


def weighted_mae_loss(pred: torch.Tensor, target: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    return (torch.abs(pred - target) * weights).sum() / weights.sum().clamp_min(1.0)

## DataLoaders and seasonal-naive benchmark

In [ ]:
train_ds = WindowDataset(values, holiday_flags, series_store_idx, series_dept_idx, CONFIG["input_weeks"], CONFIG["validation_weeks"], split_pos)
val_ds = ValidationDataset(values, holiday_flags, series_store_idx, series_dept_idx, CONFIG["input_weeks"], CONFIG["validation_weeks"], split_pos)
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

actual_val = values[:, split_pos:split_pos + CONFIG["validation_weeks"]]
seasonal_naive = values[:, split_pos - 52:split_pos - 52 + CONFIG["validation_weeks"]]
val_holidays_matrix = np.tile(holiday_flags[split_pos:split_pos + CONFIG["validation_weeks"]], (values.shape[0], 1))
seasonal_naive_wmae = wmae(actual_val.ravel(), seasonal_naive.ravel(), val_holidays_matrix.ravel(), CONFIG["holiday_weight"])

print({
    "train_windows": len(train_ds),
    "validation_series": len(val_ds),
    "seasonal_naive_wmae": seasonal_naive_wmae,
})

## Train

In [ ]:
def evaluate_model(model: nn.Module, loader: DataLoader, device: str):
    model.eval()
    pred_batches, target_batches, weight_batches = [], [], []
    norm_loss_total = 0.0
    norm_weight_total = 0.0
    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(device)
            y = batch["y"].to(device)
            weights = batch["weights"].to(device)
            series_idx = batch["series_idx"].to(device)
            store_idx = batch["store_idx"].to(device)
            dept_idx = batch["dept_idx"].to(device)
            pred_norm = model(x, series_idx, store_idx, dept_idx)
            norm_loss_total += float((torch.abs(pred_norm - y) * weights).sum().cpu())
            norm_weight_total += float(weights.sum().cpu())

            mean = batch["mean"].to(device).unsqueeze(1)
            std = batch["std"].to(device).unsqueeze(1)
            pred = (pred_norm * std + mean).clamp_min(0.0)
            target = y * std + mean

            pred_batches.append(pred.cpu().numpy())
            target_batches.append(target.cpu().numpy())
            weight_batches.append(batch["weights"].cpu().numpy())

    preds = np.concatenate(pred_batches, axis=0)
    targets = np.concatenate(target_batches, axis=0)
    weights = np.concatenate(weight_batches, axis=0)
    return {
        "preds": preds,
        "targets": targets,
        "weights": weights,
        "normalized_wmae": norm_loss_total / max(norm_weight_total, 1.0),
        "wmae": float(np.sum(np.abs(preds - targets) * weights) / np.sum(weights)),
    }


run = wandb.init(
    project=CONFIG["wandb_project"],
    entity=CONFIG["wandb_entity"],
    group=CONFIG["wandb_group"],
    job_type="experiment_train",
    name="dlinear_v4_52w_store_dept_embeddings",
    config=CONFIG,
)

wandb.config.update({
    "n_series": len(sales_panel),
    "n_stores": len(store_values),
    "n_depts": len(dept_values),
    "train_windows": len(train_ds),
    "validation_start": str(val_dates.min().date()),
    "validation_end": str(val_dates.max().date()),
    "seasonal_naive_wmae": seasonal_naive_wmae,
}, allow_val_change=True)

device = CONFIG["device"]
model = DLinearIdentity(
    seq_len=CONFIG["input_weeks"],
    pred_len=CONFIG["validation_weeks"],
    n_series=len(sales_panel),
    n_stores=len(store_values),
    n_depts=len(dept_values),
    store_embedding_dim=CONFIG["store_embedding_dim"],
    dept_embedding_dim=CONFIG["dept_embedding_dim"],
    identity_hidden_dim=CONFIG["identity_hidden_dim"],
    moving_avg_kernel=CONFIG["moving_avg_kernel"],
    use_series_calibration=CONFIG["use_series_calibration"],
).to(device)

main_params, series_params, identity_params = [], [], []
for name, param in model.named_parameters():
    if "series_bias" in name:
        series_params.append(param)
    elif "store_embedding" in name or "dept_embedding" in name or "identity_head" in name:
        identity_params.append(param)
    else:
        main_params.append(param)

optimizer = torch.optim.AdamW([
    {"params": main_params, "weight_decay": CONFIG["weight_decay"]},
    {"params": series_params, "weight_decay": CONFIG["series_bias_weight_decay"]},
    {"params": identity_params, "weight_decay": CONFIG["identity_weight_decay"]},
], lr=CONFIG["learning_rate"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=4)

best_wmae = math.inf
best_epoch = -1
best_state = None
epochs_without_improvement = 0

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    train_loss_total = 0.0
    train_weight_total = 0.0
    for batch in train_loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        weights = batch["weights"].to(device)
        series_idx = batch["series_idx"].to(device)
        store_idx = batch["store_idx"].to(device)
        dept_idx = batch["dept_idx"].to(device)

        optimizer.zero_grad(set_to_none=True)
        pred = model(x, series_idx, store_idx, dept_idx)
        loss = weighted_mae_loss(pred, y, weights)
        loss.backward()
        if CONFIG["clip_grad_norm"]:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["clip_grad_norm"])
        optimizer.step()

        train_loss_total += float((torch.abs(pred.detach() - y) * weights).sum().cpu())
        train_weight_total += float(weights.sum().cpu())

    train_loss = train_loss_total / max(train_weight_total, 1.0)
    val_result = evaluate_model(model, val_loader, device)
    scheduler.step(val_result["wmae"])

    metrics = {
        "epoch": epoch,
        "train/normalized_wmae_loss": train_loss,
        "validation/normalized_wmae_loss": val_result["normalized_wmae"],
        "validation/wmae": val_result["wmae"],
        "validation/improvement_vs_seasonal_naive_pct": 100.0 * (seasonal_naive_wmae - val_result["wmae"]) / seasonal_naive_wmae,
        "validation/improvement_vs_baseline_pct": 100.0 * (CONFIG["baseline_dlinear_39w_wmae"] - val_result["wmae"]) / CONFIG["baseline_dlinear_39w_wmae"],
        "validation/improvement_vs_v1_pct": 100.0 * (CONFIG["dlinear_v1_calibration_wmae"] - val_result["wmae"]) / CONFIG["dlinear_v1_calibration_wmae"],
        "learning_rate": optimizer.param_groups[0]["lr"],
    }
    wandb.log(metrics)
    print(metrics)

    if val_result["wmae"] < best_wmae:
        best_wmae = val_result["wmae"]
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= CONFIG["patience"]:
            print(f"Early stopping at epoch {epoch}; best epoch was {best_epoch}.")
            break

model.load_state_dict(best_state)
best_val_result = evaluate_model(model, val_loader, device)
wandb.summary["best_epoch"] = best_epoch
wandb.summary["best_validation_wmae"] = best_val_result["wmae"]
wandb.summary["seasonal_naive_wmae"] = seasonal_naive_wmae
wandb.summary["baseline_dlinear_39w_wmae"] = CONFIG["baseline_dlinear_39w_wmae"]
wandb.summary["dlinear_v1_calibration_wmae"] = CONFIG["dlinear_v1_calibration_wmae"]
wandb.summary["dlinear_v3_gated_calendar_wmae"] = CONFIG["dlinear_v3_gated_calendar_wmae"]
wandb.summary["best_improvement_vs_v1_pct"] = 100.0 * (CONFIG["dlinear_v1_calibration_wmae"] - best_val_result["wmae"]) / CONFIG["dlinear_v1_calibration_wmae"]

print({"best_epoch": best_epoch, "best_validation_wmae": best_val_result["wmae"]})

## Save artifacts

In [ ]:
preds = best_val_result["preds"]
targets = best_val_result["targets"]

records = []
for row_idx, (store, dept) in enumerate(sales_panel.index):
    for horizon_idx, date in enumerate(val_dates):
        records.append({
            "Store": int(store),
            "Dept": int(dept),
            "Date": pd.Timestamp(date),
            "IsHoliday": bool(holiday_by_date.loc[date]),
            "Weekly_Sales": float(targets[row_idx, horizon_idx]),
            "Prediction": float(preds[row_idx, horizon_idx]),
            "AbsError": float(abs(targets[row_idx, horizon_idx] - preds[row_idx, horizon_idx])),
        })

val_pred_df = pd.DataFrame(records)
val_pred_path = OUTPUT_DIR / "dlinear_v4_validation_predictions.csv"
val_pred_df.to_csv(val_pred_path, index=False)

checkpoint_path = OUTPUT_DIR / "dlinear_v4_checkpoint.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "config": CONFIG,
    "best_epoch": best_epoch,
    "best_validation_wmae": best_val_result["wmae"],
    "series_index": sales_panel.index.tolist(),
    "store_to_idx": store_to_idx,
    "dept_to_idx": dept_to_idx,
    "train_dates": [str(pd.Timestamp(d).date()) for d in all_train_dates],
    "validation_dates": [str(pd.Timestamp(d).date()) for d in val_dates],
}, checkpoint_path)

summary = {
    "experiment": "dlinear_v4_52w_store_dept_embeddings",
    "best_epoch": int(best_epoch),
    "best_validation_wmae": float(best_val_result["wmae"]),
    "baseline_dlinear_39w_wmae": float(CONFIG["baseline_dlinear_39w_wmae"]),
    "dlinear_v1_calibration_wmae": float(CONFIG["dlinear_v1_calibration_wmae"]),
    "dlinear_v3_gated_calendar_wmae": float(CONFIG["dlinear_v3_gated_calendar_wmae"]),
    "seasonal_naive_wmae": float(seasonal_naive_wmae),
    "improvement_vs_v1_pct": float(100.0 * (CONFIG["dlinear_v1_calibration_wmae"] - best_val_result["wmae"]) / CONFIG["dlinear_v1_calibration_wmae"]),
    "store_embedding_dim": CONFIG["store_embedding_dim"],
    "dept_embedding_dim": CONFIG["dept_embedding_dim"],
    "identity_hidden_dim": CONFIG["identity_hidden_dim"],
}
summary_path = OUTPUT_DIR / "dlinear_v4_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

fig, ax = plt.subplots(figsize=(7, 4))
sample = val_pred_df.sample(min(5000, len(val_pred_df)), random_state=SEED)
ax.scatter(sample["Weekly_Sales"], sample["Prediction"], s=8, alpha=0.25)
max_axis = np.nanpercentile(sample[["Weekly_Sales", "Prediction"]].to_numpy(), 99)
ax.plot([0, max_axis], [0, max_axis], color="red", linewidth=1)
ax.set_title("DLinear v4 validation predictions")
ax.set_xlabel("Actual Weekly_Sales")
ax.set_ylabel("Prediction")
plt.tight_layout()
plot_path = OUTPUT_DIR / "dlinear_v4_validation_scatter.png"
fig.savefig(plot_path, dpi=160)
plt.show()

artifact = wandb.Artifact("dlinear-v4-52w-store-dept-embeddings", type="model")
artifact.add_file(str(checkpoint_path))
artifact.add_file(str(summary_path))
artifact.add_file(str(val_pred_path))
artifact.add_file(str(plot_path))
run.log_artifact(artifact, aliases=["experiment-v4", "latest"])

wandb.log({
    "validation/prediction_table": wandb.Table(dataframe=val_pred_df.sample(min(20000, len(val_pred_df)), random_state=SEED)),
    "validation/scatter": wandb.Image(str(plot_path)),
})

summary

In [ ]:
run.finish()